# PCA + 4-Nearest Neighbour — K49-MNIST

Based on the community leaderboard submission by **dzisandy** which achieves **86.80% test accuracy** on K49 — the highest classical ML result on the leaderboard, beating the Tuned RBF SVM (85.61%).

| Model | K49 Test Accuracy | Source |
|---|---|---|
| HOG + Linear SVM v3 (ours) | 80.07% | This project |
| 4-NN raw pixels | 83.65% | K49 paper |
| Tuned RBF SVM raw pixels | 85.61% | Leaderboard |
| **PCA + 4-KNN (this notebook)** | **86.80%** | Leaderboard |
| Keras Simple CNN | 89.25% | K49 paper |
| PreActResNet-18 | 96.64% | K49 paper |

### Why this works better than HOG + SVM
Raw pixels preserve the full image signal. HOG compresses images into gradient histograms which loses fine stroke detail that turns out to be discriminative for Hiragana. PCA then removes redundant background dimensions, leaving the SVM or KNN with clean, meaningful features.

### Pipeline
```
Raw Pixels (784) → Normalize /255 → PCA (60 components) → 4-KNN → Evaluate
```

### ⚠️ Speed Warning
KNN stores all training data and computes distances at prediction time — it has **no training phase** but prediction is slow. We will benchmark inference speed in Section 9 before committing to deployment.

## 1. Import Libraries

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC, LinearSVC
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

print('✅ All libraries imported.')

✅ All libraries imported.


## 2. Load Data

In [2]:
X_train_raw  = np.load('../data/k49-train-imgs.npz')['arr_0']
y_train      = np.load('../data/k49-train-labels.npz')['arr_0']
X_test_raw   = np.load('../data/k49-test-imgs.npz')['arr_0']
y_test       = np.load('../data/k49-test-labels.npz')['arr_0']

classmap     = pd.read_csv('../data/k49_classmap.csv')
target_names = [str(classmap.iloc[i]['char']) for i in range(len(classmap))]

print(f'Training images  : {X_train_raw.shape}')
print(f'Test images      : {X_test_raw.shape}')
print(f'Number of classes: {len(np.unique(y_train))}')

Training images  : (232365, 28, 28)
Test images      : (38547, 28, 28)
Number of classes: 49


## 3. Preprocessing — Normalize and Flatten

Following the leaderboard implementation exactly:
- Divide by 255 to normalize to [0, 1]
- Flatten 28×28 → 784-dim vector
- **No StandardScaler** — confirmed harmful for image pixel data
- **No HOG** — raw pixels outperform HOG on this dataset

In [3]:
# Normalize and flatten — exactly as in the leaderboard code
X_train_flat = X_train_raw.reshape(-1, 784).astype('float32') / 255.0
X_test_flat  = X_test_raw.reshape(-1, 784).astype('float32') / 255.0

print(f'Train : {X_train_flat.shape}  dtype: {X_train_flat.dtype}')
print(f'Test  : {X_test_flat.shape}  dtype: {X_test_flat.dtype}')
print(f'Memory: {X_train_flat.nbytes / 1024**3:.2f} GiB')
print('✅ Preprocessing complete.')

Train : (232365, 784)  dtype: float32
Test  : (38547, 784)  dtype: float32
Memory: 0.68 GiB
✅ Preprocessing complete.


## 4. PCA — 60 Components

Following the leaderboard exactly: **n_components=60**, no scaler, fit only on training data.

60 components is surprisingly small — the leaderboard author found this works better than more components, likely because extra components start encoding noise rather than meaningful stroke structure.

In [4]:
# print('⏳ Fitting PCA (60 components) on full training set...')
# t0 = time.time()

# pca = PCA(n_components=60, random_state=0)   # exact leaderboard settings
# X_train_pca = pca.fit_transform(X_train_flat)  # fit on training only
# X_test_pca  = pca.transform(X_test_flat)        # apply to test

# variance_explained = pca.explained_variance_ratio_.sum() * 100
# elapsed = time.time() - t0

# print(f'  Done in {elapsed:.1f}s')
# print(f'  Input dims       : 784')
# print(f'  Output dims      : 60')
# print(f'  Variance retained: {variance_explained:.1f}%')
# print(f'  Train shape      : {X_train_pca.shape}')
# print('✅ PCA complete.')

In [5]:
# # --- Visualize PCA components as eigenfaces/eigencharacters ---
# fig, axes = plt.subplots(3, 10, figsize=(15, 5))
# for i, ax in enumerate(axes.flat):
#     ax.imshow(pca.components_[i].reshape(28, 28), cmap='RdBu_r')
#     ax.set_title(f'PC{i+1}', fontsize=7)
#     ax.axis('off')
# plt.suptitle('Top 30 PCA Components (Eigencharacters)', fontsize=12)
# plt.tight_layout()
# plt.show()

# # Cumulative variance
# cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
# plt.figure(figsize=(7, 3))
# plt.plot(range(1, 61), cumvar, marker='o', markersize=3, color='steelblue')
# plt.axhline(variance_explained, color='red', linestyle='--',
#             label=f'{variance_explained:.1f}% at 60 components')
# plt.xlabel('Number of PCA Components')
# plt.ylabel('Cumulative Variance (%)')
# plt.title('PCA Explained Variance')
# plt.legend()
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

## 5. Train KNN

KNN has **no training phase** — `fit()` simply stores all the data in memory. The work happens at prediction time when it searches for the 4 nearest neighbours.

Settings match the leaderboard exactly:
- `n_neighbors=4`
- `weights='distance'` — closer neighbours vote with higher weight
- `n_jobs=-1` — use all CPU cores for parallel distance computation

In [6]:
# clf = KNeighborsClassifier(n_neighbors=4, weights='distance', n_jobs=-1)

# print(f'⏳ Fitting KNN on {len(y_train):,} PCA-reduced samples...')
# t0 = time.time()
# clf.fit(X_train_pca, y_train)
# fit_time = time.time() - t0

# print(f'✅ Done in {fit_time:.2f}s (KNN just stores data — no real training)')
# print(f'   Memory used by model: {X_train_pca.nbytes / 1024**2:.1f} MB')

## 6. ⚡ Inference Speed Test — CRITICAL

This is the most important cell in the notebook. KNN prediction requires computing distances to all 232,365 training samples for every query. We test this **before** committing to KNN as the deployment model.

**Constraint: < 100ms per sample for real-time deployment.**

In [7]:
# # Test on a small batch first to estimate speed
# N_SPEED_TEST = 100
# X_speed_test = X_test_pca[:N_SPEED_TEST]

# print(f'⏳ Speed test: predicting {N_SPEED_TEST} samples...')
# t0 = time.time()
# _ = clf.predict(X_speed_test)
# elapsed = time.time() - t0

# ms_per_sample = elapsed / N_SPEED_TEST * 1000
# meets_constraint = ms_per_sample < 100

# print(f'\n  Total time for {N_SPEED_TEST} samples : {elapsed:.2f}s')
# print(f'  Time per sample                  : {ms_per_sample:.2f} ms')
# print(f'  Real-time constraint (<100ms)    : {"✅ MEETS" if meets_constraint else "❌ FAILS"}')

# if meets_constraint:
#     print('\n✅ KNN is fast enough for deployment. Continuing...')
# else:
#     print('\n⚠️  KNN is too slow for real-time deployment.')
#     print('   Fallback: use Tuned RBF SVM (Section 10) instead.')

## 7. Evaluate KNN on Test Set

Using the same **mean per-class accuracy** as the leaderboard to get comparable numbers.

⚠️ Predicting all 38,547 test images will take a while — grab a coffee.

In [8]:
# print(f'⏳ Predicting on {len(y_test):,} test samples...')
# t0 = time.time()
# y_test_pred     = clf.predict(X_test_pca)
# test_infer_time = (time.time() - t0) / len(y_test) * 1000
# total_pred_time = time.time() - t0

# # Standard accuracy
# test_acc_standard = accuracy_score(y_test, y_test_pred)

# # Mean per-class accuracy (leaderboard metric)
# per_class_accs = []
# for cls in range(49):
#     mask    = (y_test == cls)
#     cls_acc = (y_test_pred[mask] == cls).mean()
#     per_class_accs.append(cls_acc)
# test_acc_classavg = np.mean(per_class_accs)

# test_f1 = f1_score(y_test, y_test_pred, average='macro')

# print('=' * 45)
# print('  KNN Test Set Results')
# print('=' * 45)
# print(f'  Standard Accuracy        : {test_acc_standard*100:.2f}%')
# print(f'  Mean Per-Class Accuracy  : {test_acc_classavg*100:.2f}%  ← leaderboard metric')
# print(f'  Macro F1-Score           : {test_f1:.4f}')
# print(f'  Inference time/sample    : {test_infer_time:.2f} ms')
# print(f'  Total prediction time    : {total_pred_time:.1f}s')
# print(f'  Real-time ready          : {"✅ YES" if test_infer_time < 100 else "❌ NO"}')
# print('=' * 45)
# print(f'\n  Leaderboard target       : 86.80%')
# print(f'  Gap to leaderboard       : {(test_acc_classavg - 0.8680)*100:+.2f}%')

## 8. Full Model Comparison

In [9]:
# print('=' * 65)
# print(f'{"Model":<32} {"Test Acc (class avg)":>20} {"Source":>10}')
# print('=' * 65)
# print(f'{"HOG + Linear SVM v3 (ours)":<32} {"80.07%":>20} {"this project":>10}')
# print(f'{"4-NN raw pixels":<32} {"83.65%":>20} {"paper":>10}')
# print(f'{"Tuned RBF SVM raw pixels":<32} {"85.61%":>20} {"leaderboard":>10}')
# print(f'{"PCA + 4-KNN (this notebook)":<32} {test_acc_classavg*100:>19.2f}% {"leaderboard":>10}')
# print(f'{"Keras Simple CNN":<32} {"89.25%":>20} {"paper":>10}')
# print(f'{"PreActResNet-18":<32} {"96.64%":>20} {"paper":>10}')
# print('=' * 65)

## 9. Per-Class Analysis

In [10]:
# # Per-class accuracy bar chart
# plt.figure(figsize=(18, 5))
# colors = ['#d9534f' if a < 0.75 else '#f0ad4e' if a < 0.85 else '#5cb85c'
#           for a in per_class_accs]
# plt.bar(target_names, [a*100 for a in per_class_accs], color=colors)
# plt.axhline(test_acc_classavg*100, color='navy', linestyle='--',
#             label=f'Mean: {test_acc_classavg*100:.2f}%')
# plt.axhline(86.80, color='orange', linestyle=':', label='Leaderboard: 86.80%')
# plt.title('Per-Class Accuracy — PCA + KNN', fontsize=13)
# plt.xlabel('Hiragana Character')
# plt.ylabel('Accuracy (%)')
# plt.xticks(fontsize=9)
# plt.legend()
# plt.tight_layout()
# plt.show()

# # Worst performing classes
# worst = sorted(enumerate(per_class_accs), key=lambda x: x[1])[:10]
# print('10 Hardest Characters:')
# print(f'{"Char":>8}  {"Accuracy":>10}')
# print('-' * 22)
# for idx, acc in worst:
#     print(f'{target_names[idx]:>8}  {acc*100:>9.2f}%')

## 10. Confusion Matrix

In [11]:
# cm      = confusion_matrix(y_test, y_test_pred)
# cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

# plt.figure(figsize=(18, 15))
# sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
#             xticklabels=target_names, yticklabels=target_names)
# plt.title('Normalized Confusion Matrix — PCA + KNN', fontsize=14)
# plt.xlabel('Predicted Character', fontsize=12)
# plt.ylabel('True Character', fontsize=12)
# plt.xticks(fontsize=8, rotation=45)
# plt.yticks(fontsize=8, rotation=0)
# plt.tight_layout()
# plt.show()

In [12]:
# confused_pairs = [
#     (cm_norm[i, j], target_names[i], target_names[j])
#     for i in range(len(target_names))
#     for j in range(len(target_names))
#     if i != j
# ]
# confused_pairs.sort(reverse=True)

# print('Top 10 Most Confused Character Pairs (True -> Predicted):')
# print(f'{"True":>10} -> {"Predicted":>10}  |  Confusion Rate')
# print('-' * 45)
# for rate, true_c, pred_c in confused_pairs[:10]:
#     print(f'{true_c:>10} -> {pred_c:>10}  |  {rate:.4f} ({rate*100:.2f}%)')

## 11. Deployment Decision

Based on the speed test in Section 6, we decide which model to deploy.

In [13]:
# print('=' * 55)
# print('  Deployment Decision')
# print('=' * 55)
# print(f'  KNN inference time : {test_infer_time:.2f} ms/sample')
# print(f'  Constraint         : < 100 ms')
# print()
# if test_infer_time < 100:
#     print('  ✅ KNN meets the constraint.')
#     print('  → Deploy PCA + KNN as primary model.')
#     print(f'  → Test accuracy: {test_acc_classavg*100:.2f}% (class avg)')
# else:
#     print('  ❌ KNN is too slow for real-time deployment.')
#     print('  → Fall back to Linear SVM v3 (0.007 ms/sample)')
#     print('  → Or use Tuned RBF SVM from leaderboard (Section 12)')
# print('=' * 55)

## 12. Fallback — Tuned RBF SVM (if KNN too slow)

If KNN fails the speed constraint, this cell trains the **leaderboard Tuned RBF SVM** using the pre-found parameters C=4.15, gamma=0.00678 on raw pixels — no search needed.

⚠️ Full training on 232,365 samples takes **more than 1 day** on CPU (as noted by the leaderboard author). We use a **100,000-sample stratified subset** as a practical compromise — still significantly more data than our previous RBF attempt (50k).

In [14]:
# Only run this cell if KNN failed the speed test
RUN_FALLBACK_SVM = True

if RUN_FALLBACK_SVM:
    from sklearn.utils import resample
    import threading
    from tqdm import tqdm

    # Leaderboard pre-tuned parameters — no search needed
    C_rbf     = 4.152705171689228
    gamma_rbf = 0.006783091541660457
    SUBSET    = 100_000

    # Stratified subset of raw pixels (no HOG, no PCA for RBF)
    X_rbf, y_rbf = resample(
        X_train_flat, y_train,
        random_state=42, stratify=y_train
    )

    rbf_svm = SVC(
        C=C_rbf, gamma=gamma_rbf,
        kernel='rbf',
        decision_function_shape='ovr',
        probability=False,
        random_state=42
    )

    print(f'⏳ Training Tuned RBF SVM on full datase...')
    print(f'   C={C_rbf:.4f}, gamma={gamma_rbf:.6f}')

    t0   = time.time()
    done = threading.Event()

    def train_rbf():
        rbf_svm.fit(X_rbf, y_rbf)
        done.set()

    thread = threading.Thread(target=train_rbf)
    thread.start()

    from tqdm import tqdm
    with tqdm(desc='Training RBF SVM', bar_format='{desc} | {elapsed} elapsed') as pbar:
        while not done.is_set():
            pbar.update(0)
            done.wait(timeout=0.5)

    thread.join()
    rbf_train_time = time.time() - t0
    print(f'✅ Done in {rbf_train_time/60:.1f} min')

    # Evaluate RBF SVM
    t0 = time.time()
    y_rbf_pred      = rbf_svm.predict(X_test_flat)
    rbf_infer_time  = (time.time() - t0) / len(y_test) * 1000

    rbf_per_class = []
    for cls in range(49):
        mask = (y_test == cls)
        rbf_per_class.append((y_rbf_pred[mask] == cls).mean())
    rbf_acc_classavg = np.mean(rbf_per_class)
    rbf_f1 = f1_score(y_test, y_rbf_pred, average='macro')

    print('=' * 45)
    print('  Tuned RBF SVM Results (Fallback)')
    print('=' * 45)
    print(f'  Mean Per-Class Accuracy : {rbf_acc_classavg*100:.2f}%')
    print(f'  Macro F1-Score          : {rbf_f1:.4f}')
    print(f'  Inference time/sample   : {rbf_infer_time:.2f} ms')
    print(f'  Real-time ready         : {"✅ YES" if rbf_infer_time < 100 else "❌ NO"}')
    print('=' * 45)

    os.makedirs('../models', exist_ok=True)
    joblib.dump(rbf_svm, '../models/tuned_rbf_svm.pkl')
    print('✅ Tuned RBF SVM saved.')

else:
    print('KNN met the speed constraint — fallback SVM not needed.')

⏳ Training Tuned RBF SVM on full datase...
   C=4.1527, gamma=0.006783


Training RBF SVM | 46:42 elapsed


✅ Done in 46.7 min
  Tuned RBF SVM Results (Fallback)
  Mean Per-Class Accuracy : 83.60%
  Macro F1-Score          : 0.8434
  Inference time/sample   : 43.36 ms
  Real-time ready         : ✅ YES
✅ Tuned RBF SVM saved.


## 13. Save KNN Model

In [15]:
# os.makedirs('../models', exist_ok=True)

# joblib.dump(clf, '../models/pca_knn.pkl')
# joblib.dump(pca, '../models/pca_knn_pca.pkl')

# print('✅ Models saved:')
# print('   ../models/pca_knn.pkl      — KNN classifier')
# print('   ../models/pca_knn_pca.pkl  — PCA transformer')
# print()
# print('Inference pipeline for deployment:')
# print('   raw_pixels → /255 → flatten → pca.transform → clf.predict')

### Quick Reload

In [16]:
# Uncomment to reload on subsequent sessions
# clf = joblib.load('../models/pca_knn.pkl')
# pca = joblib.load('../models/pca_knn_pca.pkl')
# X_test_pca = pca.transform(X_test_raw.reshape(-1,784).astype('float32')/255.0)
# print('Reloaded — continue from Section 7.')

---
## Summary

### Key findings

| Finding | Evidence |
|---|---|
| HOG hurts on K49 | Raw pixels (83.65%) beat HOG+SVM (80.07%) even with nearest-neighbour |
| PCA improves KNN | PCA+KNN (86.80%) beats plain KNN (83.65%) by 3.15% |
| 60 components is enough | Leaderboard confirmed — more components adds noise not signal |
| Speed is the deployment risk | KNN trades training speed for prediction cost |

### Deployment recommendation
- **If KNN < 100ms** → Deploy PCA + KNN — highest classical accuracy at 86.80%
- **If KNN ≥ 100ms** → Deploy Tuned RBF SVM on raw pixels — still beats HOG+SVM
- **Either way** → Linear SVM v3 (80.07%) is retired as the primary model